# Statistics for Decision Making- Assignment

## Australian Real Estate Dataset 

In [40]:
#Import the libraries
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import binom, ttest_1samp, ttest_ind
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


In [45]:
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

In [4]:
#Load the dataset
df = pd.read_csv('property.csv')

In [6]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Month'] = df['Date'].dt.month
df['Year']  = df['Date'].dt.year

In [7]:
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}  →  {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Dataset loaded successfully!
Shape: (13580, 23)  →  13580 rows, 23 columns


,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount,Month,Year
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,2016-12-03,2.5,3067.0,...,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0,12,2016
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,2016-02-04,2.5,3067.0,...,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0,2,2016
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,2017-03-04,2.5,3067.0,...,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0,3,2017
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,2017-03-04,2.5,3067.0,...,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0,3,2017
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,2016-06-04,2.5,3067.0,...,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0,6,2016


In [8]:
# ── Quick overview of the dataset ────────────────────────────────────────────
print("Column names:", df.columns.tolist())
print()
print("Data types:")
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isnull().sum())


Column names: ['Suburb', 'Address', 'Rooms', 'Type', 'Price', 'Method', 'SellerG', 'Date', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'YearBuilt', 'CouncilArea', 'Lattitude', 'Longtitude', 'Regionname', 'Propertycount', 'Month', 'Year']

Data types:
Suburb                   object
Address                  object
Rooms                     int64
Type                     object
Price                   float64
Method                   object
SellerG                  object
Date             datetime64[ns]
Distance                float64
Postcode                float64
Bedroom2                float64
Bathroom                float64
Car                     float64
Landsize                float64
BuildingArea            float64
YearBuilt               float64
CouncilArea              object
Lattitude               float64
Longtitude              float64
Regionname               object
Propertycount           float64
Month                     int32
Year  

## QUestion 1:  Suburb ALtona Property Price


### Question - To test if the mean price of property has increased from $800,000

#### H0 = avg price is $800,000 & H1 = Avg price is > $800,000. 
### One-Sample T-test will be used here

In [10]:
#filter only ALtona and drop missing values
altona = df[df['Suburb'] == 'Altona']['Price'].dropna()


In [11]:
#checking the avg property price for altona suburb
print(f"Number of properties in Altona : {len(altona)}")
print(f"Sample Mean Price              : ${altona.mean():,.2f}")
print(f"Sample Std Deviation           : ${altona.std():,.2f}")

Number of properties in Altona : 74
Sample Mean Price              : $834,830.41
Sample Std Deviation           : $291,546.05


In [15]:
# One-sample t-test and 5% significance
t_stat, p_value = ttest_1samp(altona, popmean=800000, alternative='greater')

print(f"\nTest Statistic (t)             : {t_stat:.4f}")
print(f"p-value (one-tailed)           : {p_value:.4f}")
alpha = 0.05
print(f"α = {alpha}")

if p_value < alpha:
    print("REJECT H₀ → There is significant evidence the price has INCREASED above $800,000.")
else:
    print("FAIL TO REJECT H₀ → Not enough evidence that price has increased above $800,000.")



Test Statistic (t)             : 1.0277
p-value (one-tailed)           : 0.1537
α = 0.05
FAIL TO REJECT H₀ → Not enough evidence that price has increased above $800,000.


## Question 2: 2016 summer vs winter property price.


### Check if property prices differ between summer and winter months in 2016. 
Significance = 5%

 H0 = avg summer price = avg winter price
 
 H1 = avg summer price not equal to winter price
 

In [16]:
## two-sample t-test (as here 2 groups are compared)
df_2016 = df[df['Year'] == 2016].copy()

# Label each row as Summer or Winter
winter_months = [10, 11, 12, 1, 2, 3]
df_2016['Season'] = df_2016['Month'].apply(
    lambda m: 'Winter' if m in winter_months else 'Summer'
)


In [18]:
#separate into summer and winter groups
summer_prices = df_2016[df_2016['Season'] == 'Summer']['Price'].dropna()
winter_prices = df_2016[df_2016['Season'] == 'Winter']['Price'].dropna()

print(f"Summer properties (2016) : {len(summer_prices)}, Mean = ${summer_prices.mean():,.2f}")
print(f"Winter properties (2016) : {len(winter_prices)}, Mean = ${winter_prices.mean():,.2f}")

Summer properties (2016) : 4036, Mean = $1,048,054.73
Winter properties (2016) : 2300, Mean = $1,116,647.59


In [19]:
#two-sample independent t-test
t_stat, p_value = ttest_ind(summer_prices, winter_prices)

print(f"\nTest Statistic (t) : {t_stat:.4f}")
print(f"p-value (two-tailed): {p_value:.4f}")
alpha = 0.05
print(f" α = {alpha}")
if p_value < alpha:
    print("REJECT H₀ → There IS a significant difference in prices between summer and winter.")
else:
    print("FAIL TO REJECT H₀ → No significant difference in prices between seasons.")


Test Statistic (t) : -4.0434
p-value (two-tailed): 0.0001
 α = 0.05
REJECT H₀ → There IS a significant difference in prices between summer and winter.


## Question 3: Suburb of Abbotsford probablity of having a car parking space



### Find the probability that exactly 3 out of 10 properties in Abbotsford have No car parking

In [21]:
abbotsford = df[df['Suburb'] == 'Abbotsford'].copy()
abbotsford_car = abbotsford['Car'].dropna()

# Calculate probability of a property having NO car space
total_properties = len(abbotsford_car)
no_car_count     = (abbotsford_car == 0).sum()
p_no_car         = no_car_count / total_properties

print(f"Total Abbotsford properties (with Car data) : {total_properties}")
print(f"Properties with NO car parking              : {no_car_count}")
print(f"Probability of no car parking (p)           : {p_no_car:.4f}")

# Binomial distribution: n=10, k=3, p=p_no_car
n = 10
k = 3
prob = binom.pmf(k, n, p_no_car)

print(f"\nP(3 out of 10 have no car parking) = {prob:.3f}")


Total Abbotsford properties (with Car data) : 56
Properties with NO car parking              : 15
Probability of no car parking (p)           : 0.2679

P(3 out of 10 have no car parking) = 0.260


## Question 4: Probability 3 rooms in Abbotsford Suburb




### Find the probability that of a random selected property in Abbotsford has 3 rooms

In [25]:
total = len(abbotsford)
three_rooms = (abbotsford['Rooms'] == 3).sum()
p_3rooms = three_rooms / total

print(f"Total properties in Abbotsford : {total}")
print(f"Properties with 3 rooms        : {three_rooms}")
print(f"P(3 rooms)                     : {p_3rooms: .3f}")

Total properties in Abbotsford : 56
Properties with 3 rooms        : 20
P(3 rooms)                     :  0.357


## Question 5 - Probability of having 2 bathrooms in Abbostford suburb property

In [27]:
abbotsford_bath = abbotsford['Bathroom'].dropna()

total_bath  = len(abbotsford_bath)
two_bath    = (abbotsford_bath == 2).sum()
p_2bath     = two_bath / total_bath

print(f"Total properties (with Bathroom data) : {total_bath}")
print(f"Properties with 2 bathrooms           : {two_bath}")
print(f"P(2 bathrooms)                        : {p_2bath:.3f}")


Total properties (with Bathroom data) : 56
Properties with 2 bathrooms           : 19
P(2 bathrooms)                        : 0.339


## QUestion 6: One-Sample Hypothesis Test (Industry Pricing)



### Test whether actual avg is different that the stated $1,000,000 prperty price in Richmond 

### H0 -> avg = $1,000,000
### H1 -> avg is not $1,000,000

In [32]:

richmond = df[df['Suburb'] == 'Richmond']['Price'].dropna()

print(f"Number of Richmond properties : {len(richmond)}")
print(f"Sample Mean Price             : ${richmond.mean():,.3f}")
print(f"Sample Std Deviation          : ${richmond.std():,.3f}")

# Two-tailed one-sample t-test
t_stat, p_value = ttest_1samp(richmond, popmean=1_000_000)

print(f"\nNull Hypothesis (H₀)  : µ = $1,000,000")
print(f"Alternative (H₁)      : µ ≠ $1,000,000")
print(f"Test Statistic (t)    : {t_stat:.4f}")
print(f"p-value (two-tailed)  : {p_value:.4f}")

alpha = 0.05
print(f"α = {alpha}")
if p_value < alpha:
    print("\nREJECT H₀ → The actual average price in Richmond is significantly DIFFERENT from $1,000,000.")
    if richmond.mean() > 1_000_000:
        print(f"The mean (${richmond.mean():,.2f}) is HIGHER than the claimed $1,000,000.")
else:
    print("\nFAIL TO REJECT H₀ → The data does not contradict the firm's claim of avg price = $1,000,000.")

print("\nBusiness Conclusion:")
print("The real estate firm's claim of $1,000,000 avg property price in richmond is inaccurate.")
print(f"Properties in Richmond are priced higher and the average is calculated to be ${richmond.mean():,.2f}.")


Number of Richmond properties : 260
Sample Mean Price             : $1,083,564.423
Sample Std Deviation          : $522,353.525

Null Hypothesis (H₀)  : µ = $1,000,000
Alternative (H₁)      : µ ≠ $1,000,000
Test Statistic (t)    : 2.5795
p-value (two-tailed)  : 0.0104
α = 0.05

REJECT H₀ → The actual average price in Richmond is significantly DIFFERENT from $1,000,000.
The mean ($1,083,564.42) is HIGHER than the claimed $1,000,000.

Business Conclusion:
The real estate firm's claim of $1,000,000 avg property price in richmond is inaccurate.
Properties in Richmond are priced higher and the average is calculated to be $1,083,564.42.


## Question 7: Independent Two-Sample T-Test (Feature Impact) 



### Do properties with car parking sell at a higher average price than properties without car parking, across the entire dataset?

### H0 = avg price of property (w car parking) LESS than or EUQAL to avg price of property (NO car parking) 
### H1 = avg price of property (w car parking) is GREATER then avg price of property (NO car parking) 

In [37]:
car_yes = df[df['Car'] >  0]['Price'].dropna()  
car_no  = df[df['Car'] == 0]['Price'].dropna()

print(f"Properties WITH car parking    : {len(car_yes)}, Mean = ${car_yes.mean():,.2f}")
print(f"Properties WITHOUT car parking : {len(car_no)},  Mean = ${car_no.mean():,.2f}")

# t-test: testing if car_yes > car_no
t_stat, p_value = ttest_ind(car_yes, car_no, alternative='greater')

print(f"\nTest Statistic (t)  : {t_stat:.3f}")
print(f"p-value (one-tailed): {p_value:.3f}")
alpha = 0.05
print(f"α = {alpha}")
if p_value < alpha:
    print("\nREJECT H₀ → Properties WITH car parking sell at HIGHER prices.")
else:
    print("\nFAIL TO REJECT H₀ → Car parking DOES NOT increase price.")

print("\nBusiness Implication for Developers:")
print("Car parking does NOT guarantee a higher sale price. \nDevelopers should consider the construction cost of parking spcae as per market demand for parking space.")


Properties WITH car parking    : 12492, Mean = $1,074,443.92
Properties WITHOUT car parking : 1026,  Mean = $1,079,088.01

Test Statistic (t)  : -0.223
p-value (one-tailed): 0.588
α = 0.05

FAIL TO REJECT H₀ → Car parking DOES NOT increase price.

Business Implication for Developers:
Car parking does NOT guarantee a higher sale price. 
Developers should consider the construction cost of parking spcae as per market demand for parking space.


## Quesiton 8:  Two-Way ANOVA (Location & Property Type) 


### TO find if the property prices are affected by Location, type of property, Interaction between suburb and property type

In [46]:

#top 5 suburbs
top5_suburbs = df['Suburb'].value_counts().head(5).index.tolist()
print("Top 5 suburbs selected:", top5_suburbs)

df_anova = df[df['Suburb'].isin(top5_suburbs)].dropna(subset=['Price', 'Type'])
print(f"Records used for ANOVA: {len(df_anova)}")
print(f"Property types present: {df_anova['Type'].unique()}")

model = ols('Price ~ C(Suburb) + C(Type) + C(Suburb):C(Type)', data=df_anova).fit()
anova_table = anova_lm(model, typ=2)

print("\n===== Two-Way ANOVA Results =====")
print(anova_table.round(4))

alpha = 0.05
print("\n--- Interpretation ---")
for factor in ['C(Suburb)', 'C(Type)', 'C(Suburb):C(Type)']:
    p = anova_table.loc[factor, 'PR(>F)']
    label = factor.replace('C(','').replace(')','')
    if p < alpha:
        print(f"  {label}: p={p:.4f} → SIGNIFICANT effect on price ✓")
    else:
        print(f"  {label}: p={p:.4f} → NOT significant")


Top 5 suburbs selected: ['Reservoir', 'Richmond', 'Bentleigh East', 'Preston', 'Brunswick']
Records used for ANOVA: 1329
Property types present: ['h' 't' 'u']

===== Two-Way ANOVA Results =====
                         sum_sq      df         F  PR(>F)
C(Suburb)          4.160952e+13     4.0  139.1428     0.0
C(Type)            6.480908e+13     2.0  433.4449     0.0
C(Suburb):C(Type)  6.129412e+12     8.0   10.2484     0.0
Residual           9.823523e+13  1314.0       NaN     NaN

--- Interpretation ---
  Suburb: p=0.0000 → SIGNIFICANT effect on price ✓
  Type: p=0.0000 → SIGNIFICANT effect on price ✓
  Suburb:Type: p=0.0000 → SIGNIFICANT effect on price ✓


## Question 9:  p-Value Interpretation (Decision Making) 

### A hypothesis test comparing prices across two suburbs results in a p-value of 0.032. 

Tasks: 

### • What does this p-value indicate? 
A p-value below 0.05 means that the probability of difference in the price is 95%. This is significant and confirms that the pricing across two suburbs differ. 



### • Should the null hypothesis be rejected at α = 0.05? 
At 5% significance this hypothesis (H0) should be rejected as the given p-value (0.032) < 0.05.

### • How should a business stakeholder interpret this result? 
A buisness stakeholder should take this into accound as the price difference is statistically significant. This will affect the pricing strategies, investment decisions, contruction costs and valueation of the property for selling. 

## Question 10:  Industry-Style Hypothesis Validation (Policy Decision) 

### A housing policy group believes that properties with more than 2 bathrooms command a premium price. 
Design and execute a statistical test to validate this claim: 

• Report p-value 

• Give a clear recommendation to policymakers 

Here, 

H0 = avg price of house (> 2 bathrrom) <= avg price of house (< 2 baths)

H1 = avg price of house (> 2 bathrrom) > avg price of house (< 2 bathrrom)

In [50]:
gt2_bath = df[df['Bathroom'] >  2]['Price'].dropna() #> 2 abths
le2_bath  = df[df['Bathroom'] <= 2]['Price'].dropna() # < 2 baths

print(f"Properties with >2 bathrooms : {len(gt2_bath)}, Mean = ${gt2_bath.mean():,.2f}")
print(f"Properties with ≤2 bathrooms : {len(le2_bath)}, Mean = ${le2_bath.mean():,.2f}")


Properties with >2 bathrooms : 1060, Mean = $1,882,824.20
Properties with ≤2 bathrooms : 12520, Mean = $1,007,347.94


In [51]:
#t-test
t_stat, p_value = ttest_ind(gt2_bath, le2_bath, alternative='greater')

print(f"Test Statistic (t)    : {t_stat:.3f}")
print(f"p-value (one-tailed)  : {p_value:.3f}")

alpha = 0.05
print(f"α = {alpha}")
if p_value < alpha:
    print("REJECT H₀ → Properties with MORE than 2 bathrooms DO command a premium price.")
else:
    print("FAIL TO REJECT H₀ → No significant premium for extra bathrooms.")

print("\nRecommendation to Policymakers:")
print("  Properties with more than 2 bathrooms are priced significantly higher.")
print(f"  The average premium is ${gt2_bath.mean() - le2_bath.mean():,.2f} more than ≤2 bathroom properties.")
print("  Housing affordability policies should account for bathroom count as a price driver.")
print("  Affordable housing programs should target smaller-bathroom stock.")

Test Statistic (t)    : 46.026
p-value (one-tailed)  : 0.000
α = 0.05
REJECT H₀ → Properties with MORE than 2 bathrooms DO command a premium price.

Recommendation to Policymakers:
  Properties with more than 2 bathrooms are priced significantly higher.
  The average premium is $875,476.26 more than ≤2 bathroom properties.
  Housing affordability policies should account for bathroom count as a price driver.
  Affordable housing programs should target smaller-bathroom stock.
